<a href="https://colab.research.google.com/github/rodrigorissettoterra/SENAI_Concepcao_e_Design_de_ML/blob/main/Keras_e_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 1 — Keras: Classificador de câncer de mama

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Carregar dataset

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

No sklearn:
- 0 = maligno
- 1 = benigno

Separar treino e teste

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Padronizar os dados

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Criar modelo Keras

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compilar modelo

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

Treinar modelo

In [7]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.6841 - loss: 0.6022 - val_accuracy: 0.8242 - val_loss: 0.5024
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8956 - loss: 0.3921 - val_accuracy: 0.9011 - val_loss: 0.3380
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9313 - loss: 0.2780 - val_accuracy: 0.9560 - val_loss: 0.2470
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9478 - loss: 0.2139 - val_accuracy: 0.9670 - val_loss: 0.1947
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9505 - loss: 0.1748 - val_accuracy: 0.9670 - val_loss: 0.1593
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9615 - loss: 0.1481 - val_accuracy: 0.9890 - val_loss: 0.1356
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9698 - loss: 0.1304 - val_accuracy: 0.9890 - val_loss: 0.1192
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9698 - loss: 0.1172 - val_accuracy: 0.9890 - va

Avaliar

In [8]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Acurácia no teste:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9737 - loss: 0.1031 
Acurácia no teste: 0.9736841917037964


Previsões

In [9]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

Matriz de confusão:
[[40  2]
 [ 1 71]]

Relatório de classificação:
              precision    recall  f1-score   support

   malignant       0.98      0.95      0.96        42
      benign       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



# Etapa 2 — PyTorch

Base PyTorch para classificação binária

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


def train_binary_model(X, y, epochs=100, lr=0.001):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

    model = BinaryClassifier(X_train.shape[1])

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()

        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(f"Época {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

    model.eval()

    with torch.no_grad():
        y_pred_prob = model(X_test_tensor)
        y_pred = (y_pred_prob >= 0.5).int().numpy()

    print("\nAcurácia:", accuracy_score(y_test, y_pred))
    print("\nMatriz de confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("\nRelatório:")
    print(classification_report(y_test, y_pred))

    return model, scaler

Caso 1 — PyTorch com Iris Dataset

Classificação binária:
- 1 = Iris setosa
- 0 = Não setosa

In [11]:
from sklearn.datasets import load_iris

iris = load_iris()

X_iris = iris.data
y_iris = (iris.target == 0).astype(int)

modelo_iris, scaler_iris = train_binary_model(
    X_iris,
    y_iris,
    epochs=100,
    lr=0.001
)

Época 20/100 - Loss: 0.5653
Época 40/100 - Loss: 0.5005
Época 60/100 - Loss: 0.4272
Época 80/100 - Loss: 0.3526
Época 100/100 - Loss: 0.2833

Acurácia: 0.9666666666666667

Matriz de confusão:
[[20  0]
 [ 1  9]]

Relatório:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        20
           1       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.98      0.95      0.96        30
weighted avg       0.97      0.97      0.97        30



Caso 2 — PyTorch com câncer de mama

In [12]:
from sklearn.datasets import load_breast_cancer

breast = load_breast_cancer()

X_breast = breast.data
y_breast = breast.target

modelo_cancer, scaler_cancer = train_binary_model(
    X_breast,
    y_breast,
    epochs=100,
    lr=0.001
)

Época 20/100 - Loss: 0.6478
Época 40/100 - Loss: 0.5151
Época 60/100 - Loss: 0.3824
Época 80/100 - Loss: 0.2784
Época 100/100 - Loss: 0.2065

Acurácia: 0.9122807017543859

Matriz de confusão:
[[38  4]
 [ 6 66]]

Relatório:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88        42
           1       0.94      0.92      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.91      0.91       114
weighted avg       0.91      0.91      0.91       114



Caso 3 — PyTorch com Marketing Bancário

O dataset Bank Marketing da UCI tem como objetivo prever se o cliente irá assinar um depósito a prazo, variável y, com valores yes ou no

In [13]:
import pandas as pd
import zipfile
import requests
from io import BytesIO

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

response = requests.get(url)
zip_file = zipfile.ZipFile(BytesIO(response.content))

print(zip_file.namelist())

df_bank = pd.read_csv(zip_file.open("bank-full.csv"), sep=";")

df_bank.head()

['bank-full.csv', 'bank-names.txt', 'bank.csv']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


Preparar os dados

In [14]:
# Separar variáveis explicativas e variável-alvo
X_bank = df_bank.drop("y", axis=1)
y_bank = df_bank["y"]

# Converter alvo:
# no = 0
# yes = 1
y_bank = y_bank.map({
    "no": 0,
    "yes": 1
}).values

# Converter variáveis categóricas em variáveis numéricas
X_bank = pd.get_dummies(X_bank, drop_first=True)

# Converter para array
X_bank = X_bank.values

print("Formato de X:", X_bank.shape)
print("Formato de y:", y_bank.shape)

Formato de X: (45211, 42)
Formato de y: (45211,)


Treinar o modelo PyTorch

In [15]:
modelo_bank, scaler_bank = train_binary_model(
    X_bank,
    y_bank,
    epochs=100,
    lr=0.001
)

Época 20/100 - Loss: 0.7245
Época 40/100 - Loss: 0.6842
Época 60/100 - Loss: 0.6272
Época 80/100 - Loss: 0.5433
Época 100/100 - Loss: 0.4454

Acurácia: 0.8830034280659074

Matriz de confusão:
[[7985    0]
 [1058    0]]

Relatório:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94      7985
           1       0.00      0.00      0.00      1058

    accuracy                           0.88      9043
   macro avg       0.44      0.50      0.47      9043
weighted avg       0.78      0.88      0.83      9043



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Caso 4 — PyTorch com Fraudes V2

- Class = 0 → Transação normal
- Class = 1 → Fraude

Fazer upload do arquivo

In [16]:
import pandas as pd

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df_fraude = pd.read_csv(url)

df_fraude.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Preparar os dados

In [17]:
X_fraude = df_fraude.drop("Class", axis=1)
y_fraude = df_fraude["Class"].values

# Converter para numérico (já está, mas mantemos padrão)
X_fraude = X_fraude.values

print(X_fraude.shape, y_fraude.shape)

(284807, 30) (284807,)


Treinar com PyTorch

In [18]:
modelo_fraude, scaler_fraude = train_binary_model(
    X_fraude,
    y_fraude,
    epochs=50,
    lr=0.001
)

Época 20/50 - Loss: 0.7659
Época 40/50 - Loss: 0.6917

Acurácia: 0.7486921105298269

Matriz de confusão:
[[42646 14218]
 [   97     1]]

Relatório:
              precision    recall  f1-score   support

           0       1.00      0.75      0.86     56864
           1       0.00      0.01      0.00        98

    accuracy                           0.75     56962
   macro avg       0.50      0.38      0.43     56962
weighted avg       1.00      0.75      0.85     56962



In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Precision: 0.9726027397260274
Recall: 0.9861111111111112
F1: 0.9793103448275862
